# Notebook 3 — Visualisations & Intervention Map

## Objective
Produce all submission visualisations and the prioritised intervention map for the bottom 10 districts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/clean/analysis_ready.csv')
anomalies = pd.read_csv('../data/clean/bottom10_anomalies.csv')
print(f"Loaded {len(df)} districts, {len(anomalies)} anomalies")


In [ ]:
# State-level summary
state_avg = df.groupby('state').agg(
    avg_uptake=('uptake_rate','mean'),
    avg_gap=('gap_rate','mean'),
    n_districts=('district','count')
).sort_values('avg_gap', ascending=False)
print("State summary (sorted by avg gap):")
print(state_avg.head(10).round(3).to_string())


In [ ]:
# Scatter: SC/ST share vs uptake (most significant predictor)
fig, ax = plt.subplots(figsize=(9, 5))
normal = df[df['residual'] > df['residual'].quantile(0.10)]
anom = df[df['is_anomaly'] == True] if 'is_anomaly' in df.columns else anomalies

ax.scatter(normal['sc_st_share']*100, normal['uptake_rate']*100,
           alpha=0.3, color='#4a90c4', s=18)
ax.scatter(anom['sc_st_share']*100, anom['uptake_rate']*100,
           color='#e05a2b', s=70, zorder=5, edgecolors='white', linewidth=0.5, label='Target districts')
for _, row in anom.head(5).iterrows():
    ax.annotate(row['state'][:6], (row['sc_st_share']*100, row['uptake_rate']*100),
                fontsize=7, xytext=(4,3), textcoords='offset points', color='#e05a2b')
m, b = np.polyfit(df['sc_st_share'], df['uptake_rate'], 1)
xs = np.linspace(df['sc_st_share'].min(), df['sc_st_share'].max(), 100)
ax.plot(xs*100, (m*xs+b)*100, '--', color='#1a3a5c', alpha=0.5, linewidth=1.5)
ax.set_xlabel('SC/ST population share (%)'); ax.set_ylabel('PM-KISAN uptake rate (%)')
ax.set_title('SC/ST share vs PM-KISAN uptake\n(Significant negative correlation, p<0.05)', fontsize=11)
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('../outputs/02_covariate_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Intervention map table
def assign_intervention(row):
    if row['female_literacy'] < 45:
        return 'Women SHG-led enrollment drives', 'Female literacy below 45%'
    elif row['sc_st_share'] > 0.40:
        return 'CSO partnerships + Gram Sabha drives', 'High SC/ST population with exclusion barriers'
    elif row['internet_access'] < 0.25:
        return 'Offline Aadhaar/Jan Dhan linkage camps', 'Low internet — digital channels failing'
    else:
        return 'IEC campaign + panchayat drives', 'General awareness deficit'

anomalies[['intervention','rationale']] = anomalies.apply(
    lambda r: pd.Series(assign_intervention(r)), axis=1)

print("PRIORITISED INTERVENTION MAP — BOTTOM 10 DISTRICTS")
print("="*80)
for rank, (_, row) in enumerate(anomalies.iterrows(), 1):
    print(f"\n#{rank} | {row['state']} — {row['district']}")
    print(f"   Uptake: {row['uptake_rate']*100:.1f}%  |  Gap: {row['gap_rate']*100:.1f}%  |  Residual: {row['residual']:.3f}")
    print(f"   Intervention: {row['intervention']}")
    print(f"   Rationale:    {row['rationale']}")

anomalies.to_csv('../outputs/intervention_map.csv', index=False)
print("\nSaved: ../outputs/intervention_map.csv ✓")


In [ ]:
# Bottom 10 bar chart
fig, ax = plt.subplots(figsize=(10, 6))
top10 = anomalies.sort_values('gap_rate')
colors = ['#e05a2b' if g > 0.55 else '#d4a017' for g in top10['gap_rate']]
bars = ax.barh(range(len(top10)), top10['gap_rate']*100, color=colors, height=0.6)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels([f"{r['state'][:10]} — {r['district'].split('_')[1]}"
                    for _, r in top10.iterrows()], fontsize=9)
for i, (_, val) in enumerate(zip(bars, top10['gap_rate']*100)):
    ax.text(val+0.5, i, f"{val:.1f}%", va='center', fontsize=8.5)
ax.set_xlabel('Uptake gap (%)'); ax.axvline(50, color='#aaa', linestyle=':')
ax.set_title('10 districts with worst unexplained uptake gap', fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig('../outputs/03_bottom10_districts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Notebook 3 complete ✓")
